# Эксперименты

Будем обучать LogReg, kNN, дерево решений, рандомный лес и градиентный бустинг. Возможно, ещё попробуем полносвязные нейронные сети. Основная метрика - ROC-AUC

In [4]:
import sys, os, json
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')
import pandas as pd
from src.modeling import load_splits, run_experiments
train, val, test = load_splits()

In [5]:
results, best = run_experiments(train, val, test)
df_exp = pd.DataFrame(results)
df_exp

[logreg] cv_auc=0.6221  val_auc=0.6397  params={'C': 0.01, 'penalty': 'l2'}
[knn] cv_auc=0.6769  val_auc=0.7314  params={'n_neighbors': 31, 'weights': 'distance'}
[decision_tree] cv_auc=0.6911  val_auc=0.7364  params={'max_depth': 8, 'min_samples_leaf': 5}
[random_forest] cv_auc=0.7186  val_auc=0.7608  params={'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 200}
[gbdt] cv_auc=0.7089  val_auc=0.7660  params={'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100}
[ensemble_voting] cv_auc=0.7039  val_auc=0.7529  params={}

[BEST=gbdt] TEST metrics: {'roc_auc': 0.7061829652996845, 'accuracy': 0.6518163125428376, 'f1_macro': 0.6413707008341882, 'log_loss': 0.6154654470097813}


,model,best_params,cv_roc_auc,val_roc_auc,val_accuracy,val_f1_macro,val_log_loss
0,logreg,"{'C': 0.01, 'penalty': 'l2'}",0.622129,0.639674,0.598355,0.588923,0.661110
1,knn,"{'n_neighbors': 31, 'weights': 'distance'}",0.676899,0.731421,0.666210,0.661051,0.608529
2,decision_tree,"{'max_depth': 8, 'min_samples_leaf': 5}",0.691128,0.736421,0.679232,0.660467,0.717461
3,random_forest,"{'max_depth': 8, 'min_samples_leaf': 1, 'n_est...",0.718571,0.760813,0.687457,0.672580,0.596602
4,gbdt,"{'learning_rate': 0.05, 'max_depth': 4, 'n_est...",0.708858,0.765976,0.699794,0.694560,0.584618
5,ensemble_voting,{},0.703888,0.752930,0.679232,0.668961,0.607460


In [8]:
print('Best model on test:')
print(best)

Best model on test:
{'name': 'gbdt', 'test': {'roc_auc': 0.7061829652996845, 'accuracy': 0.6518163125428376, 'f1_macro': 0.6413707008341882, 'log_loss': 0.6154654470097813}}


### Выводы:
1. В эксперименты был включен `VotingClassifier` (soft voting), объединяющий логистическую регрессию, случайный лес и градиентный бустинг. Это классический подход для бинарной классификации, позволяющий снизить дисперсию.
2. Лучшие результаты среди базовых моделей и ансамблей показывает **Gradient Boosting**. В отличие от линейных моделей (LogReg), GBDT эффективно выявляет нелинейные зависимости и сложные взаимодействия между признаками игроков и команд. 
В отличие от kNN, деревья не страдают от «проклятия размерности» и шума в данных. В отличие от Random Forest, бустинг итеративно корректирует ошибки, что дает лучшую точность при небольшой глубине деревьев (depth=4), защищая модель от переобучения.
3. Итоговый пайплайн с фича-инжинирингом и GBDT дает значительный прирост по ROC-AUC (до 0.76+) по сравнению с бейзлайном (0.50), подтверждая качество проделанной работы с данными.